## Importing Libraries

In [2]:
#new
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio

from geomstats.geometry.hypersphere import Hypersphere
from geomstats.geometry.euclidean import Euclidean
from geomstats.learning.frechet_mean import FrechetMean

# Latent space and prior
import torch
import torch.nn as nn
import torch.optim as optim


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type == "cuda":
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("CPU name:", torch.get_num_threads(), "threads")

sigma = 0.35
n = 10000

# Reproducibility: fix seeds for deterministic behavior
SEED = 5
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
import random
random.seed(SEED)

INFO: Note: NumExpr detected 40 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
INFO: NumExpr defaulting to 16 threads.


Using device: cuda
GPU name: NVIDIA RTX A6000


## Define Sphere and Softplus

In [3]:
sphere = Hypersphere(dim=2)          # S^2 embedded in R^3
metric = sphere.metric

# Base point mu: north pole
mu = np.array([0.0, 0.0, 1.0])

def softplus(x):
    return np.log1p(np.exp(x))

## True Decoder

In [4]:
# Decoder parameters (from your confirmation)
# w1 = np.array([3.0, -2.0])          # shape (2,)
# b1 = np.array([1.0, 3.0])           # shape (2,)

# w2 = np.array([[0.05, -0.5],
#                [-0.15, -0.1]])       # shape (2, 2)

# b2 = np.array([0.062, 0.609])        # shape (2,)


w1 = np.array([2.5, -2.0])          # shape (2,)
b1 = np.array([1.5, 3.0])           # shape (2,)

w2 = np.array([[0.05, -0.4],
               [-0.16, -0.1]])       # shape (2, 2)

b2 = np.array([0.07, 0.6])        # shape (2,)


def true_decoder(z):
    """
    z: shape (n,)
    returns: shape (n, 2)
    """
    z = z.reshape(-1, 1)             # (n, 1)
    h = softplus(z * w1 + b1)         # (n, 2)
    return h @ w2.T + b2              # (n, 2)

## Sample the Latent and Map to Sphere

In [5]:
# Sample latent variables
z = np.random.normal(0.0, 1.0, size=n)

# Decode into tangent vectors (R^2)
v_2d = true_decoder(z)

# Embed tangent vectors into R^3 (T_mu S^2)
v_3d = np.zeros((n, 3))
v_3d[:, :2] = v_2d

# Exponential map to the sphere
m = metric.exp(v_3d, base_point=mu)

In [6]:
# Norms should be ~1
print("Mean norm:", np.mean(np.linalg.norm(m, axis=1)))
print("Min norm:", np.min(np.linalg.norm(m, axis=1)))
print("Max norm:", np.max(np.linalg.norm(m, axis=1)))

Mean norm: 1.0
Min norm: 0.9999999999999997
Max norm: 1.0000000000000002


## Visualize on Sphere

In [7]:
# Unit sphere surface
u = np.linspace(0, 2 * np.pi, 60)
v = np.linspace(0, np.pi, 30)

xs = np.outer(np.cos(u), np.sin(v))
ys = np.outer(np.sin(u), np.sin(v))
zs = np.outer(np.ones_like(u), np.cos(v))

fig = go.Figure()


fig.add_trace(
    go.Scatter3d(
        x=m[:, 0],
        y=m[:, 1],
        z=m[:, 2],
        mode="markers",
        marker=dict(
            size=3,
            opacity=0.8,
            color=z,
            colorscale="Viridis",
            colorbar=dict(title="z")
        ),
        name="Synthetic data"
    )
)


fig.add_trace(
    go.Surface(
        x=xs,
        y=ys,
        z=zs,
        opacity=0.15,
        colorscale="Greys",
        showscale=False,
        name="Unit sphere"
    )
)


fig.update_layout(
    font=dict(size=16),
    title="Synthetic data on S^2",
    scene=dict(
        xaxis=dict(title="x"),
        yaxis=dict(title="y"),
        zaxis=dict(title="z"),
        aspectmode="data"
    ),
    width=700,
    height=700,
    showlegend=False
)

# fig.write_image("sphere_plot.png", width=700, height=700, scale=2)
fig.show()

## Riemannian Gaussian noise on the sphere

In [7]:
np.random.seed(SEED)  # deterministic Riemannian sampling
precision = 1.0 / (sigma**2)

# geomstats samples around ONE mean at a time, so do a small loop (fine for n ~ 1e3)
x = np.zeros_like(m)
for i in range(n):
    x[i] = sphere.random_riemannian_normal(mean=m[i], precision=precision, n_samples=1)

# quick sanity checks
m_norm_err = np.max(np.abs(np.linalg.norm(np.array(m), axis=1) - 1.0))
x_norm_err = np.max(np.abs(np.linalg.norm(np.array(x), axis=1) - 1.0))
d = np.array([sphere.metric.dist(m[i], x[i]) for i in range(n)])

print("m shape:", np.array(m).shape, "x shape:", np.array(x).shape)
print("max | ||m||-1 |:", m_norm_err)
print("max | ||x||-1 |:", x_norm_err)
print("mean geodesic dist(m,x):", float(d.mean()), "std:", float(d.std()))

m shape: (10000, 3) x shape: (10000, 3)
max | ||m||-1 |: 3.3306690738754696e-16
max | ||x||-1 |: 3.3306690738754696e-16
mean geodesic dist(m,x): 0.4311225892694793 std: 0.22747062195086434


## Plotting on Sphere: clean points (m) + noisy points (x)

In [8]:
uu = np.linspace(0, 2*np.pi, 50)
vv = np.linspace(0, np.pi, 25)
xs = np.outer(np.cos(uu), np.sin(vv))
ys = np.outer(np.sin(uu), np.sin(vv))
zs = np.outer(np.ones_like(uu), np.cos(vv))

fig = go.Figure()

fig.add_trace(go.Surface(x=xs, y=ys, z=zs, opacity=0.15, showscale=False))

fig.add_trace(go.Scatter3d(
    x=np.array(m)[:, 0], y=np.array(m)[:, 1], z=np.array(m)[:, 2],
    mode="markers", marker=dict(size=2),
    name="clean means m"
))

fig.add_trace(go.Scatter3d(
    x=np.array(x)[:, 0], y=np.array(x)[:, 1], z=np.array(x)[:, 2],
    mode="markers", marker=dict(size=2),
    name="noisy samples x"
))

fig.update_layout(
    font=dict(size=16),
    title="Clean submanifold points + Riemannian Gaussian noise",
    scene=dict(aspectmode="data")
)

# fig.write_image("noisy_sphere_plot.png", width=700, height=700, scale=2)

## Compute the empirical Fréchet mean (mu_hat)

In [9]:
from geomstats.learning.frechet_mean import FrechetMean

frechet = FrechetMean(space=sphere)
frechet.fit(x)

mu_hat = frechet.estimate_

print("Estimated base point mu_hat:", mu_hat)
print("||mu_hat||:", np.linalg.norm(mu_hat))

Estimated base point mu_hat: [-0.88644648 -0.0977609   0.4523886 ]
||mu_hat||: 1.0


## Define the RVAE model

In [10]:
latent_dim = 1

In [11]:
# Encoder
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3, 16)
        self.fc2 = nn.Linear(16, 16)
        self.mean = nn.Linear(16, 1)
        self.logvar = nn.Linear(16, 1)

    def forward(self, x):
        h = torch.nn.functional.softplus(self.fc1(x))
        h = torch.nn.functional.softplus(self.fc2(h))
        return self.mean(h), self.logvar(h)

In [12]:
# Decoder
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(1, 2)
        self.fc2 = nn.Linear(2, 2)

    def forward(self, z):
        h = torch.nn.functional.softplus(self.fc1(z))
        return self.fc2(h)  # tangent vectors (R^2)

In [13]:
def sphere_exp(mu, v, eps=1e-9):
    # mu: (3,) on device, assumed unit norm (numpy or tensor)
    # v:  (B,3) tangent at mu (so dot(v,mu)=0 ideally) (numpy or tensor)
    mu = torch.as_tensor(mu, dtype=torch.float32, device=v.device if isinstance(v, torch.Tensor) else None)
    v = torch.as_tensor(v, dtype=torch.float32, device=mu.device)
    v_norm = torch.linalg.norm(v, dim=-1, keepdim=True).clamp_min(eps)  # (B,1)
    return torch.cos(v_norm) * mu + torch.sin(v_norm) * (v / v_norm)

def sphere_dist(p, q, eps=1e-7):
    # p,q: (B,3) unit vectors
    inner = (p * q).sum(dim=-1).clamp(-1 + eps, 1 - eps)
    return torch.acos(inner)  # (B,)

In [14]:
precision = 1.0 / (sigma ** 2)

In [15]:
def tangent_basis(mu_hat_t, eps=1e-9):
    # mu_hat_t: (3,) unit vector on device
    ref = torch.tensor([1.0, 0.0, 0.0], device=mu_hat_t.device, dtype=mu_hat_t.dtype)
    if torch.abs((ref * mu_hat_t).sum()) > 0.9:
        ref = torch.tensor([0.0, 1.0, 0.0], device=mu_hat_t.device, dtype=mu_hat_t.dtype)

    e1 = ref - (ref * mu_hat_t).sum() * mu_hat_t
    e1 = e1 / torch.linalg.norm(e1).clamp_min(eps)

    e2 = torch.linalg.cross(mu_hat_t, e1)
    e2 = e2 / torch.linalg.norm(e2).clamp_min(eps)

    return e1, e2

In [16]:
def decoder_to_tangent(v2, mu_hat_t):
    # v2: (B,2)
    e1, e2 = tangent_basis(mu_hat_t)
    v = v2[:, 0:1] * e1 + v2[:, 1:2] * e2  # (B,3)

    # enforce tangency numerically (optional but recommended)
    v = v - (v * mu_hat_t).sum(dim=-1, keepdim=True) * mu_hat_t
    return v

In [17]:
def riemannian_loglikelihood_torch(x, z, decoder, mu_hat_t, precision):
    v2 = decoder(z)                 # (B,2)
    v3 = decoder_to_tangent(v2, mu_hat_t)  # (B,3) tangent at mu_hat_t
    m  = sphere_exp(mu_hat_t, v3)   # (B,3)
    d  = sphere_dist(x, m)          # (B,)
    return -0.5 * precision * (d ** 2)

In [18]:
def elbo(v_x_t, x_tensor, encoder, decoder, mu_hat_t, precision):
    mean, logvar = encoder(v_x_t)  # (B,1), (B,1)
    std = torch.exp(0.5 * logvar)
    z = mean + torch.randn_like(std) * std

    log_pxz = riemannian_loglikelihood_torch(x_tensor, z, decoder, mu_hat_t, precision).mean()

    kl_per_sample = -0.5 * (1 + logvar - mean.pow(2) - logvar.exp())  # (B,1)
    kl = kl_per_sample.mean()

    return log_pxz - kl

In [19]:
mu_hat_t = torch.tensor(mu_hat, dtype=torch.float32, device=device)
mu_hat_t = mu_hat_t / torch.linalg.norm(mu_hat_t)  # just in case

## Training Loop RVAE

In [20]:
encoder = Encoder().to(device)
decoder = Decoder().to(device)

optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-3)

x_tensor = torch.tensor(x, dtype=torch.float32, device=device)
v_x = sphere.metric.log(x, base_point=mu_hat)
print("Tangent vectors of x (log map at mu_hat): shape", v_x.shape)
v_x_t = torch.tensor(v_x, dtype=torch.float32, device=device) #tangent vectors

precision = 1.0 / (sigma ** 2)

for epoch in range(5000):
    optimizer.zero_grad()
    loss = -elbo(v_x_t, x_tensor, encoder, decoder, mu_hat_t, precision)
    loss.backward()
    optimizer.step()

    if epoch % 200 == 0:
        print(epoch, float(loss))
        #print("grad check:", decoder.fc1.weight.grad.abs().mean().item())

Tangent vectors of x (log map at mu_hat): shape (10000, 3)
0 3.851536512374878
200 3.360506772994995
400 2.4874441623687744
600 2.2169384956359863
800 2.127274513244629
1000 2.0723185539245605
1200 2.0524935722351074
1400 1.991694688796997
1600 1.9599801301956177
1800 1.9331762790679932
2000 1.9208697080612183
2200 1.905229926109314
2400 1.8980739116668701
2600 1.8943963050842285
2800 1.8898526430130005
3000 1.8792141675949097
3200 1.8816418647766113
3400 1.8810184001922607
3600 1.8752052783966064
3800 1.855006217956543
4000 1.8782408237457275
4200 1.870844841003418
4400 1.8792028427124023
4600 1.8725589513778687
4800 1.8634252548217773


## Compute the true submanifold curve

In [21]:
# 1) Choose a grid of latent values (CPU numpy for geomstats)
z_grid_np = np.linspace(-3.0, 3.0, 500)

# 2) True decoder outputs 2D tangent coordinates at mu (R^2)
v2_true = true_decoder(z_grid_np)  # shape (500, 2), numpy

# 3) Embed into T_mu S^2 (R^3 with last coord 0)
v3_true = np.zeros((len(z_grid_np), 3))
v3_true[:, :2] = v2_true

# 4) Map onto S^2 using Exp_mu
m_true = sphere.metric.exp(v3_true, mu)  # shape (500, 3), numpy

## Compute the learned submanifold curve

In [22]:
z_grid = torch.linspace(-3, 3, 500, device=device).unsqueeze(1)

with torch.no_grad():
    v2_hat = decoder(z_grid)                    # (500,2) torch
    v3_hat = decoder_to_tangent(v2_hat, mu_hat_t).cpu().numpy()
m_hat = sphere.metric.exp(v3_hat, mu_hat)       # geomstats exp is now fed true tangent vectors

## Plot: noisy samples x + true curve + learned curve + $\mu$ and $\hat{\mu}$

In [23]:
# Sphere surface for context
uu = np.linspace(0, 2*np.pi, 60)
vv = np.linspace(0, np.pi, 30)
xs = np.outer(np.cos(uu), np.sin(vv))
ys = np.outer(np.sin(uu), np.sin(vv))
zs = np.outer(np.ones_like(uu), np.cos(vv))

fig = go.Figure()

fig.add_trace(go.Surface(
    x=xs, y=ys, z=zs,
    opacity=0.12,
    showscale=False,
    name="S^2 surface"
))

# Noisy samples
fig.add_trace(go.Scatter3d(
    x=np.array(x)[:, 0], y=np.array(x)[:, 1], z=np.array(x)[:, 2],
    mode="markers",
    marker=dict(size=2, opacity=0.45),
    name="noisy samples x"
))

# True submanifold curve (clean)
fig.add_trace(go.Scatter3d(
    x=m_true[:, 0], y=m_true[:, 1], z=m_true[:, 2],
    mode="lines",
    line=dict(width=6),
    name="true submanifold m(z)"
))

# Learned submanifold curve
fig.add_trace(go.Scatter3d(
    x=m_hat[:, 0], y=m_hat[:, 1], z=m_hat[:, 2],
    mode="lines",
    line=dict(width=6),
    name="learned submanifold m_hat(z)"
))

# Base points mu and mu_hat
fig.add_trace(go.Scatter3d(
    x=[mu[0]], y=[mu[1]], z=[mu[2]],
    mode="markers+text",
    marker=dict(size=6),
    text=["mu"],
    textposition="top center",
    name="mu"
))

fig.add_trace(go.Scatter3d(
    x=[mu_hat[0]], y=[mu_hat[1]], z=[mu_hat[2]],
    mode="markers+text",
    marker=dict(size=6),
    text=["mu_hat"],
    textposition="top center",
    name="mu_hat"
))

fig.update_layout(
    font=dict(size=16),
    title="Step 2 diagnostic: true vs learned submanifold on S^2 (with noisy samples)",
    scene=dict(aspectmode="data"),
    width=800,
    height=800
)
fig.show()
# fig.write_image("sphere_curve.png", width=700, height=700, scale=2)

# VAE

## VAE Decoder

In [24]:
class DecoderVAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(1, 2)
        self.out = nn.Linear(2, 3)   # ambient R^3

    def forward(self, z):
        h = torch.nn.functional.softplus(self.fc1(z))
        return self.out(h)            # (B,3)

## Likelihood with Euclidean Gaussian

In [25]:
def euclidean_loglikelihood(x, z, decoder, sigma, eps=1e-12):
    # x: (B,3) torch
    # z: (B,1) torch
    x_hat = decoder(z)                             # (B,3)
    precision = 1.0 / (sigma**2 + eps)
    return -0.5 * precision * torch.sum((x - x_hat) ** 2, dim=-1)  # (B,)

## VAE Elbo

In [26]:
def elbo_vae(x, encoder, decoder, sigma):
    mean, logvar = encoder(x)                      # (B,1), (B,1)

    std = torch.exp(0.5 * logvar)
    z = mean + torch.randn_like(std) * std         # reparameterization

    log_pxz = euclidean_loglikelihood(x, z, decoder, sigma).mean()

    kl_per_sample = -0.5 * (1 + logvar - mean.pow(2) - logvar.exp())  # (B,1)
    kl = kl_per_sample.mean()

    return log_pxz - kl

## Training Loop VAE

In [27]:
encoder = Encoder().to(device)
decoder_vae = DecoderVAE().to(device)

optimizer = optim.Adam(
    list(encoder.parameters()) + list(decoder_vae.parameters()),
    lr=1e-3
)
x_tensor = torch.tensor(x, dtype=torch.float32, device=device)

for epoch in range(5000):
    optimizer.zero_grad()
    loss = -elbo_vae(x_tensor, encoder, decoder_vae, sigma)  # <-- use decoder_vae
    loss.backward()
    optimizer.step()

    if epoch % 200 == 0:
        print(epoch, float(loss))

0 5.965686321258545
200 2.5729100704193115
400 2.2610015869140625
600 2.116358757019043
800 2.0465774536132812
1000 2.016850471496582
1200 1.9909201860427856
1400 1.9946646690368652
1600 1.98512864112854
1800 1.9689040184020996
2000 1.9660515785217285
2200 1.9628016948699951
2400 1.9633479118347168
2600 1.9572980403900146
2800 1.945930004119873
3000 1.9435656070709229
3200 1.955366849899292
3400 1.94748055934906
3600 1.938546895980835
3800 1.9492971897125244
4000 1.9489243030548096
4200 1.9531903266906738
4400 1.9398369789123535
4600 1.9401538372039795
4800 1.928223729133606


## Compute learned VAE curve

In [28]:
# --- VAE curve on the same latent grid ---
device = next(decoder_vae.parameters()).device

z_grid_t = torch.linspace(-3, 3, 500, device=device).unsqueeze(1)

with torch.no_grad():
    x_vae = decoder_vae(z_grid_t).detach().cpu().numpy()  # (500,3) in R^3

## Tangent VAE Training

In [29]:
# Tangent vectors of x using geomstats Hypersphere
# Log map: tangent vectors at base_point (mu_hat, Fréchet mean) pointing toward each point in x
v_x = sphere.metric.log(x, base_point=mu_hat)
print("Tangent vectors of x (log map at mu_hat): shape", v_x.shape)
v_x_t = torch.tensor(v_x, dtype=torch.float32, device=device)

encoder = Encoder().to(device)
decoder_vaet = DecoderVAE().to(device)

optimizer = optim.Adam(
    list(encoder.parameters()) + list(decoder_vaet.parameters()),
    lr=1e-3
)

for epoch in range(5000):
    optimizer.zero_grad()
    loss = -elbo_vae(v_x_t, encoder, decoder_vaet, sigma)  # <-- use decoder_vaet
    loss.backward()
    optimizer.step()

    if epoch % 200 == 0:
        print(epoch, float(loss))

Tangent vectors of x (log map at mu_hat): shape (10000, 3)
0 9.929505348205566
200 4.302842140197754
400 3.4420320987701416
600 3.1921513080596924
800 2.639078140258789
1000 2.3523612022399902
1200 2.223846197128296
1400 2.1616647243499756
1600 2.129088878631592
1800 2.1142163276672363
2000 2.0999155044555664
2200 2.0861058235168457
2400 2.091183662414551
2600 2.0922861099243164
2800 2.0639357566833496
3000 2.067913055419922
3200 2.080953598022461
3400 2.0683741569519043
3600 2.058612823486328
3800 2.077418565750122
4000 2.0645792484283447
4200 2.0721397399902344
4400 2.052975654602051
4600 2.056722640991211
4800 2.067093849182129


## Tangent PCA

In [30]:
# Tangent PCA: SVD on tangent vectors at mu_hat
tangent_flat = v_x.T  # (3, n) - each column is a tangent vector
U, svals, V_t = np.linalg.svd(tangent_flat, full_matrices=False)

# PC1 direction in tangent space (first principal component)
pc1 = U[:, 0]  # (3,)

# Grid along the first principal direction (same range as other methods: -3 to 3)
z_grid_tpca = np.linspace(-3, 3, 500)

# Tangent vectors along PC1
v_tpca = z_grid_tpca[:, None] * pc1  # (500, 3)

# Exponential map to sphere -> submanifold curve
m_tpca = sphere.metric.exp(v_tpca, base_point=mu_hat)  # (500, 3)

## PCA

In [31]:
# PCA on data matrix x (Euclidean PCA)
x_centered = x - x.mean(axis=0)  # center the data
U_pca, s_pca, Vt_pca = np.linalg.svd(x_centered, full_matrices=False)

# PC1 direction
pc1_eucl = Vt_pca[0]  # (3,)

# Grid along the first principal component
z_grid_pca = np.linspace(-3, 3, 500)

# Points along PCA line in R^3
x_mean = x.mean(axis=0)
points_pca = x_mean + z_grid_pca[:, None] * pc1_eucl  # (500, 3)

m_pca = points_pca[points_pca[:,0]**2 + points_pca[:,1]**2 + points_pca[:,2]**2 < 4]

## Plotting VAE, Tangent VAE and RVAE curves

In [32]:
# Sphere surface for context
uu = np.linspace(0, 2*np.pi, 60)
vv = np.linspace(0, np.pi, 30)
xs = np.outer(np.cos(uu), np.sin(vv))
ys = np.outer(np.sin(uu), np.sin(vv))
zs = np.outer(np.ones_like(uu), np.cos(vv))

fig = go.Figure()

fig.add_trace(go.Surface(
    x=xs, y=ys, z=zs,
    opacity=0.12,
    showscale=False,
    name="S^2 surface"
))
# Noisy samples
fig.add_trace(go.Scatter3d(
    x=np.array(x)[:, 0], y=np.array(x)[:, 1], z=np.array(x)[:, 2],
    mode="markers",
    marker=dict(size=3, opacity=0.1),
    name="noisy samples x"
))

# True submanifold curve (clean)
fig.add_trace(go.Scatter3d(
    x=m_true[:, 0], y=m_true[:, 1], z=m_true[:, 2],
    mode="lines",
    line=dict(width=6, color="#C1121F"),
    name="True Submanifold"
))

# Learned submanifold curve (rVAE)
fig.add_trace(go.Scatter3d(
    x=m_hat[:, 0], y=m_hat[:, 1], z=m_hat[:, 2],
    mode="lines",
    line=dict(width=6, color='black'),
    name="KT-RSV"
))

# Learned submanifold curve (Tangent PCA)
fig.add_trace(go.Scatter3d(
    x=m_tpca[:, 0], y=m_tpca[:, 1], z=m_tpca[:, 2],
    mode="lines",
    line=dict(width=6, color="#7E57C2"),
    name="Tangent PCA"
))

# Learned submanifold curve (PCA on x, projected to S^2)
fig.add_trace(go.Scatter3d(
    x=m_pca[:, 0], y=m_pca[:, 1], z=m_pca[:, 2],
    mode="lines",
    line=dict(width=6, color="blue"),
    name="PCA"
))

# --- Tangent VAE additions ---
device = next(decoder_vaet.parameters()).device
z_grid_t = torch.linspace(-3, 3, 500, device=device).unsqueeze(1)

with torch.no_grad():
    x_vaet = decoder_vaet(z_grid_t).detach()  # Keep as tensor for sphere_exp

tvae = sphere_exp(mu_hat, x_vaet).cpu().numpy()   # (B,3) - base point mu_hat
# x_vaet = x_vaet.cpu().numpy()

# Move tangent vectors to tangent plane: μ̂ + v lies in T_μ̂ S² (affine plane at μ̂)
# x_vaet_in_plane = np.array(mu_hat) + x_vaet  # (500, 3) points in tangent plane

# fig.add_trace(go.Scatter3d(
#     x=x_vaet_in_plane[:, 0], y=x_vaet_in_plane[:, 1], z=x_vaet_in_plane[:, 2],
#     mode="lines",
#     line=dict(width=6, color="grey", dash="dash"),
#     name="Tangent Vectors (Tangent VAE)"
# ))

fig.add_trace(go.Scatter3d(
    x=tvae[:, 0], y=tvae[:, 1], z=tvae[:, 2],
    mode="lines",
    line=dict(width=6, color="#00897B"),
    name="Tangent VAE"
))


# --- VAE curve additions ---
device = next(decoder_vae.parameters()).device
z_grid_t = torch.linspace(-3, 3, 500, device=device).unsqueeze(1)

with torch.no_grad():
    x_vae = decoder_vae(z_grid_t).detach().cpu().numpy()  # (500,3) in R^3

fig.add_trace(go.Scatter3d(
    x=x_vae[:, 0], y=x_vae[:, 1], z=x_vae[:, 2],
    mode="lines",
    line=dict(width=6, color="grey"),
    name="VAE"
))


fig.update_layout(
    font=dict(size=18),
    title=dict(text="Comparison of Different Submanifold Learning Methods", 
    pad=dict(b=25)),
    scene=dict(
        aspectmode="data",
        xaxis=dict(tickfont=dict(size=14)),
        yaxis=dict(tickfont=dict(size=14)),
        zaxis=dict(tickfont=dict(size=14), 
        tickvals=[-1, -0.5, 0, 0.5, 1, 1.5, 2]),
    ),
    width=800,
    height=800,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=0.93,
        xanchor="center",
        x=0.5,
    ),
    margin=dict(l=0, r=0, t=50, b=0, pad=0),
)
# fig.write_image("result_figures/manifold_plot.png", scale=300/72)
fig.show()

In [33]:
# # Combined metric: geodesic distance + off-manifold penalty
# def geodesic_distances(a, b):
#     return np.array([sphere.metric.dist(a[i], b[i]) for i in range(len(a))])

# def off_manifold_penalty(x):
#     # distance from unit norm
#     return np.abs(np.linalg.norm(x, axis=1) - 1.0)

# # Project VAE outputs to S^2 for geodesic part
# eps = 1e-12
# x_vae_proj = x_vae / np.clip(np.linalg.norm(x_vae, axis=1, keepdims=True), eps, None)

# # rVAE (already on manifold)
# geo_rvae = geodesic_distances(m_true, m_hat)
# penalty_rvae = off_manifold_penalty(m_hat)  # should be near 0

# # VAE (off-manifold)
# geo_vae = geodesic_distances(m_true, x_vae_proj)
# penalty_vae = off_manifold_penalty(x_vae)

# # Choose penalty weight lambda
# lam = 10  # try 0.1, 1.0, 10.0 depending on how strongly you want to penalize off-manifold

# score_rvae = geo_rvae.mean() + lam * penalty_rvae.mean()
# score_vae  = geo_vae.mean()  + lam * penalty_vae.mean()

# print("=== Combined metric (geodesic + off-manifold penalty) ===")
# print(f"rVAE: geo={geo_rvae.mean():.6f}, penalty={penalty_rvae.mean():.6f}, score={score_rvae:.6f}")
# print(f" VAE: geo={geo_vae.mean():.6f}, penalty={penalty_vae.mean():.6f}, score={score_vae:.6f}")